# Lezione 3 — Pratica: supremazia e simulazione classica

In questa pratica riprendi l'esempio della catena di Ising con campo trasverso della lezione teorica e completi il confronto tra diversi numeri di ripetizioni di Trotter.

## Traccia

Hai già calcolato le curve di magnetizzazione con un passo di Trotter e con dieci passi. Il tuo compito:

1. Genera i circuiti con tre ripetizioni di Trotter (`trotter_lie(hamiltoniano, float(t), 3)`), a metà strada tra la versione a un passo e quella a dieci
2. Calcola la magnetizzazione di quei circuiti con `capa_expectativa`
3. Disegna le tre curve insieme (uno, tre e dieci passi) nello stesso grafico
4. Annota, in una frase all'interno di un commento, quale relazione osservi tra il numero di passi di Trotter e la somiglianza di ciascuna curva con la curva di riferimento (dieci passi)

### Suggerimenti

- Usa la stessa funzione `trotter_lie` che ha già generato le curve a uno e dieci passi: cambia solo il numero di ripetizioni
- Il layer `capa_expectativa` sa già valutare una lista di circuiti insieme all'operatore `magnetizacion`
- Confronta visivamente quale delle due curve, quella a uno o quella a dieci passi, è più vicina a quella a tre passi

In [ ]:
import cirq
import tensorflow_quantum as tfq
import numpy as np
import matplotlib.pyplot as plt

def cadena_ising_campo_transverso(n, campos):
    """Hamiltoniano de una cadena de n qubits con interacción Z-Z y campo X."""
    qubits = [cirq.GridQubit(0, i) for i in range(n)]
    hamiltoniano = cirq.PauliSum()
    for i in range(n - 1):
        hamiltoniano -= cirq.Z(qubits[i]) * cirq.Z(qubits[i + 1])
    for i in range(n):
        hamiltoniano -= campos[i] * cirq.X(qubits[i])
    return hamiltoniano

def paso_trotter(hamiltoniano, t):
    """Aproxima exp(-i * H * t) aplicando cada término del Hamiltoniano por separado."""
    return cirq.Circuit([tfq.util.exponential([t * termino]) for termino in hamiltoniano])

def trotter_lie(hamiltoniano, t, repeticiones):
    """Cuantas más repeticiones, más fino el paso y más preciso el resultado."""
    circuito = cirq.Circuit()
    for _ in range(repeticiones):
        circuito += paso_trotter(hamiltoniano, t / repeticiones)
    return circuito

n = 4
campos = [1.0] * n
hamiltoniano = cadena_ising_campo_transverso(n, campos)
qubits = [cirq.GridQubit(0, i) for i in range(n)]
magnetizacion = sum(cirq.Z(q) for q in qubits)

tiempos = np.linspace(0.01, 2, 20)

# Ya calculadas: las curvas de referencia de 1 y 10 repeticiones de Trotter
circuitos_1_paso = [trotter_lie(hamiltoniano, float(t), 1) for t in tiempos]
circuitos_10_pasos = [trotter_lie(hamiltoniano, float(t), 10) for t in tiempos]

capa_expectativa = tfq.layers.Expectation()
exp_1_paso = capa_expectativa(circuitos_1_paso, symbol_names=[], symbol_values=[[] for _ in tiempos], operators=magnetizacion).numpy()
exp_10_pasos = capa_expectativa(circuitos_10_pasos, symbol_names=[], symbol_values=[[] for _ in tiempos], operators=magnetizacion).numpy()

# TODO: genera los circuitos con 3 repeticiones de Trotter, igual que hiciste con 1 y 10
circuitos_3_pasos = None

# TODO: calcula la magnetización de esos circuitos con capa_expectativa
exp_3_pasos = None

# TODO: dibuja las tres curvas juntas (1, 3 y 10 pasos) con plt.plot, plt.xlabel, plt.ylabel y plt.legend

# TODO: en un comentario, anota qué relación observas entre el número de pasos de Trotter
# y el parecido de cada curva con la curva de referencia (10 pasos)

<details>
<summary>💡 Mostra soluzione</summary>

```python
# Generiamo i circuiti con 3 ripetizioni di Trotter, a metà strada tra 1 e 10
circuitos_3_pasos = [trotter_lie(hamiltoniano, float(t), 3) for t in tiempos]

# Calcoliamo la magnetizzazione di quei circuiti con lo stesso layer di TensorFlow Quantum
exp_3_pasos = capa_expectativa(circuitos_3_pasos, symbol_names=[], symbol_values=[[] for _ in tiempos], operators=magnetizacion).numpy()

# Disegniamo le tre curve insieme per confrontarle
plt.plot(tiempos, exp_1_paso, label="1 passo di Trotter")
plt.plot(tiempos, exp_3_pasos, label="3 passi di Trotter")
plt.plot(tiempos, exp_10_pasos, label="10 passi di Trotter")
plt.xlabel("Tempo")
plt.ylabel("Magnetizzazione")
plt.legend()
plt.show()

# Relazione osservata: più ripetizioni di Trotter usi, più la curva si avvicina
# a quella di riferimento (10 passi); la curva a 3 passi è chiaramente più vicina
# a quella a 10 passi di quanto lo sia quella a 1 passo, perché un passo più fine
# approssima meglio l'evoluzione esatta
```

</details>